# Моделирование: LightGBM + подбор гиперпараметров

Пайплайн:
1. Загрузка данных + анализ нулей
2. Walk-forward CV по годам
3. Optuna — поиск гиперпараметров
4. Финальная модель на лучших параметрах
5. Сравнение с baseline

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.max_colwidth', 200)

WEATHER_CSV  = 'data/weather_features.csv'
DAILY_CSV    = 'data/daily_counts.csv'
NON_FEATURES = {'date', 'report_count', 'avg_likes', 'total_photos'}
TEST_YEAR    = 2025

## 1. Загрузка данных и анализ нулей

In [ ]:
weather = pd.read_csv(WEATHER_CSV, parse_dates=['date'])
daily   = pd.read_csv(DAILY_CSV,   parse_dates=['date'])

df = weather.merge(daily[['date', 'report_count']], on='date', how='left')

nan_before = df['report_count'].isna().sum()
print(f'Дат в погоде: {len(df)}')
print(f'Из них без постов (NaN до fillna): {nan_before} ({nan_before/len(df)*100:.1f}%)')

df['report_count'] = df['report_count'].fillna(0).astype(int)
df = df[df['year'] >= 2020].reset_index(drop=True)

zero_days  = (df['report_count'] == 0).sum()
total_days = len(df)
print(f'\nДанные с 2020 года (весь год):')
print(f'  Всего дней: {total_days}')
print(f'  Дней с 0 отчётов: {zero_days} ({zero_days/total_days*100:.1f}%)')
print(f'  Дней с >=1 отчётом: {total_days - zero_days} ({(total_days-zero_days)/total_days*100:.1f}%)')
print(f'\nРаспределение report_count:')
print(df['report_count'].describe())

In [ ]:
# Доля нулей по годам
zero_by_year = df.groupby('year').apply(
    lambda g: pd.Series({
        'total': len(g),
        'zeros': (g['report_count'] == 0).sum(),
        'zero_pct': (g['report_count'] == 0).mean() * 100,
        'mean_nonzero': g.loc[g['report_count'] > 0, 'report_count'].mean(),
    })
).reset_index()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['% дней с нулём по годам', 'Среднее число отчётов (только ненулевые дни)'])

fig.add_trace(go.Bar(
    x=zero_by_year['year'], y=zero_by_year['zero_pct'],
    marker_color='#FF6F00', name='% нулей'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=zero_by_year['year'], y=zero_by_year['mean_nonzero'].round(1),
    marker_color='#2196F3', name='среднее (ненулевые)'
), row=1, col=2)

fig.update_layout(height=380, plot_bgcolor='white', showlegend=False,
    title='Структура таргета по годам')
fig.update_yaxes(gridcolor='#e0e0e0')
fig.show()
zero_by_year

In [ ]:
# Распределение report_count (только ненулевые дни, log-scale)
nonzero = df[df['report_count'] > 0]['report_count']

fig = go.Figure(go.Histogram(
    x=nonzero, nbinsx=60,
    marker_color='#2196F3', opacity=0.8,
))
fig.update_layout(
    title='Распределение числа отчётов (дни с >0 отчётами)',
    xaxis_title='report_count', yaxis_title='Дней',
    height=380, plot_bgcolor='white',
    yaxis=dict(gridcolor='#e0e0e0'),
    xaxis=dict(gridcolor='#e0e0e0'),
)
fig.show()
print(f'Медиана: {nonzero.median():.0f}, p75: {nonzero.quantile(0.75):.0f}, p95: {nonzero.quantile(0.95):.0f}, max: {nonzero.max()}')

## 2. Walk-forward кросс-валидация по годам

Обучаем на годах до N, тестируем на году N. Каждый год — отдельный фолд.
Это правильный способ CV для временных рядов — нет утечки будущего в прошлое.

In [ ]:
def get_feature_cols(df):
    return [c for c in df.columns
            if c not in NON_FEATURES and c != 'date' and df[c].dtype != object]


def eval_params(df, params, cv_years=None):
    """Walk-forward CV. Возвращает dict с метриками по фолдам и средними."""
    feature_cols = get_feature_cols(df)
    years = sorted(df['year'].unique())

    if cv_years is None:
        # Начинаем со второго года (нужен хотя бы один год на трейн)
        cv_years = [y for y in years if y >= years[1]]

    folds = []
    for test_year in cv_years:
        train_df = df[df['year'] < test_year]
        test_df  = df[df['year'] == test_year]
        if len(train_df) == 0 or len(test_df) == 0:
            continue

        X_tr, y_tr = train_df[feature_cols], train_df['report_count']
        X_te, y_te = test_df[feature_cols],  test_df['report_count']

        model = lgb.LGBMRegressor(**params, verbose=-1, random_state=42)
        model.fit(X_tr, y_tr,
                  eval_set=[(X_te, y_te)],
                  callbacks=[lgb.early_stopping(30, verbose=False)])

        preds = model.predict(X_te).clip(0)
        folds.append({
            'year': test_year,
            'rmse': np.sqrt(mean_squared_error(y_te, preds)),
            'mae':  mean_absolute_error(y_te, preds),
            'r2':   r2_score(y_te, preds),
            'n':    len(test_df),
        })

    folds_df = pd.DataFrame(folds)
    return folds_df


# Базовая линия — параметры по умолчанию
BASE_PARAMS = dict(
    n_estimators=500, learning_rate=0.05,
    max_depth=6, num_leaves=31,
    min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
)

# CV только на годах 2020-2024 (у 2018-2019 мало данных)
CV_YEARS = list(range(2020, TEST_YEAR))

print('Baseline walk-forward CV...')
baseline_cv = eval_params(df, BASE_PARAMS, cv_years=CV_YEARS)
print(baseline_cv.to_string(index=False))
print(f'\nСреднее: RMSE={baseline_cv["rmse"].mean():.3f}, MAE={baseline_cv["mae"].mean():.3f}, R2={baseline_cv["r2"].mean():.3f}')

In [ ]:
# График метрик по фолдам
fig = make_subplots(rows=1, cols=3, subplot_titles=['RMSE (меньше лучше)', 'MAE', 'R² (больше лучше)'])

for col, metric, color in [('rmse', 'RMSE', '#e6194b'), ('mae', 'MAE', '#FF6F00'), ('r2', 'R2', '#2196F3')]:
    c = ['RMSE', 'MAE', 'R2'].index(metric) + 1
    fig.add_trace(go.Bar(
        x=baseline_cv['year'].astype(str), y=baseline_cv[col].round(3),
        marker_color=color, name=metric,
        text=baseline_cv[col].round(2), textposition='outside',
    ), row=1, col=c)

fig.update_layout(height=400, plot_bgcolor='white', showlegend=False,
    title='Walk-forward CV — метрики по годам (baseline)')
fig.update_yaxes(gridcolor='#e0e0e0')
fig.show()

## 3. Подбор гиперпараметров — Optuna

Optuna использует байесовскую оптимизацию (TPE). Минимизируем среднее RMSE по walk-forward фолдам.

In [ ]:
def objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators', 200, 1500),
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 15, 127),
        max_depth         = trial.suggest_int('max_depth', 3, 10),
        min_child_samples = trial.suggest_int('min_child_samples', 5, 50),
        subsample         = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    )
    folds = eval_params(df, params, cv_years=CV_YEARS)
    return folds['rmse'].mean()


N_TRIALS = 80  # увеличь до 150-200 если есть время

study = optuna.create_study(direction='minimize',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nЛучшее RMSE: {study.best_value:.3f}')
print(f'Лучшие параметры:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# История оптимизации
trials_df = study.trials_dataframe()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=trials_df.index, y=trials_df['value'],
    mode='markers', name='RMSE попытки',
    marker=dict(color='#90CAF9', size=5),
))
# Лучшее на каждом шаге
best_so_far = trials_df['value'].cummin()
fig.add_trace(go.Scatter(
    x=trials_df.index, y=best_so_far,
    mode='lines', name='Лучшее RMSE',
    line=dict(color='#e6194b', width=2),
))
fig.update_layout(
    title='Optuna — история поиска',
    xaxis_title='Попытка', yaxis_title='RMSE (CV)',
    height=380, plot_bgcolor='white', hovermode='x unified',
    yaxis=dict(gridcolor='#e0e0e0'), xaxis=dict(gridcolor='#e0e0e0'),
)
fig.show()

In [ ]:
# Важность гиперпараметров по версии Optuna
param_imp = optuna.importance.get_param_importances(study)

fig = go.Figure(go.Bar(
    x=list(param_imp.values()),
    y=list(param_imp.keys()),
    orientation='h',
    marker_color='#2196F3',
))
fig.update_layout(
    title='Важность гиперпараметров (Optuna fANOVA)',
    xaxis_title='Относительная важность',
    yaxis=dict(autorange='reversed'),
    height=380, plot_bgcolor='white',
    xaxis=dict(gridcolor='#e0e0e0'),
)
fig.show()

## 4. Финальная модель с лучшими параметрами

In [ ]:
best_params = study.best_params.copy()

print('CV с лучшими параметрами...')
best_cv = eval_params(df, best_params, cv_years=CV_YEARS)
print(best_cv.to_string(index=False))
print(f'\nСреднее: RMSE={best_cv["rmse"].mean():.3f}, MAE={best_cv["mae"].mean():.3f}, R2={best_cv["r2"].mean():.3f}')

In [ ]:
# Сравнение baseline vs лучшие параметры
compare = pd.DataFrame([
    {'model': 'Baseline', 'rmse': baseline_cv['rmse'].mean(), 'mae': baseline_cv['mae'].mean(), 'r2': baseline_cv['r2'].mean()},
    {'model': 'Optuna',   'rmse': best_cv['rmse'].mean(),    'mae': best_cv['mae'].mean(),    'r2': best_cv['r2'].mean()},
])

fig = make_subplots(rows=1, cols=3, subplot_titles=['RMSE', 'MAE', 'R²'])
colors = ['#90CAF9', '#2196F3']

for col, metric in [('rmse', 'RMSE'), ('mae', 'MAE'), ('r2', 'R²')]:
    c = ['RMSE', 'MAE', 'R²'].index(metric) + 1
    fig.add_trace(go.Bar(
        x=compare['model'], y=compare[col].round(3),
        marker_color=colors,
        text=compare[col].round(3), textposition='outside',
        name=metric,
    ), row=1, col=c)

fig.update_layout(height=400, plot_bgcolor='white', showlegend=False,
    title='Baseline vs Optuna — среднее по CV фолдам')
fig.update_yaxes(gridcolor='#e0e0e0')
fig.show()
compare

In [ ]:
# Финальная модель: трейн 2018-2024, тест 2025
feature_cols = get_feature_cols(df)

train_df = df[df['year'] < TEST_YEAR]
test_df  = df[df['year'] == TEST_YEAR]

X_tr, y_tr = train_df[feature_cols], train_df['report_count']
X_te, y_te = test_df[feature_cols],  test_df['report_count']

final_model = lgb.LGBMRegressor(**best_params, verbose=-1, random_state=42)
final_model.fit(X_tr, y_tr,
                eval_set=[(X_te, y_te)],
                callbacks=[lgb.early_stopping(50, verbose=False)])

preds_test  = final_model.predict(X_te).clip(0)
preds_train = final_model.predict(X_tr).clip(0)

rmse = np.sqrt(mean_squared_error(y_te, preds_test))
mae  = mean_absolute_error(y_te, preds_test)
r2   = r2_score(y_te, preds_test)
r2_tr = r2_score(y_tr, preds_train)

print(f'Финальная модель ({TEST_YEAR}):')
print(f'  RMSE: {rmse:.3f}')
print(f'  MAE:  {mae:.3f}')
print(f'  R2 (test):  {r2:.3f}')
print(f'  R2 (train): {r2_tr:.3f}')

In [ ]:
# График: факт vs прогноз (трейн + тест)
train_dates = train_df['date']
test_dates  = test_df['date']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=train_dates, y=y_tr,
    mode='lines', name='Факт (трейн)',
    line=dict(color='#90CAF9', width=1.5), opacity=0.8,
))
fig.add_trace(go.Scatter(
    x=train_dates, y=preds_train,
    mode='lines', name='Прогноз (трейн)',
    line=dict(color='#EF9A9A', width=1.5, dash='dot'), opacity=0.8,
))
fig.add_vline(
    x=test_dates.min().timestamp() * 1000,
    line_dash='dash', line_color='#555', line_width=1.5,
    annotation_text=f'Тест ({TEST_YEAR})', annotation_position='top right',
)
fig.add_trace(go.Scatter(
    x=test_dates, y=y_te,
    mode='lines', name='Факт (тест)',
    line=dict(color='#2196F3', width=2),
))
fig.add_trace(go.Scatter(
    x=test_dates, y=preds_test,
    mode='lines', name='Прогноз (тест)',
    line=dict(color='#e6194b', width=2, dash='dot'),
))
fig.update_layout(
    title=f'Финальная модель (Optuna) — R² тест={r2:.3f}, трейн={r2_tr:.3f}',
    xaxis_title='Дата', yaxis_title='Отчётов',
    hovermode='x unified', height=450, plot_bgcolor='white',
    yaxis=dict(gridcolor='#e0e0e0'), xaxis=dict(gridcolor='#e0e0e0'),
    legend=dict(orientation='h', y=-0.2),
)
fig.show()

In [ ]:
# Feature importance — топ-30
importance = pd.Series(
    final_model.feature_importances_, index=feature_cols
).sort_values(ascending=False)

top_imp = importance.head(30)
fig = go.Figure(go.Bar(
    x=top_imp.values, y=top_imp.index,
    orientation='h', marker_color='#2196F3',
))
fig.update_layout(
    title='Feature Importance LightGBM Optuna (топ-30)',
    xaxis_title='Importance (split)',
    yaxis=dict(autorange='reversed'),
    height=700, plot_bgcolor='white',
    xaxis=dict(gridcolor='#e0e0e0'),
)
fig.show()

In [ ]:
# Калибровка: факт vs прогноз
from sklearn.metrics import r2_score

preds_df = test_df[['date']].copy()
preds_df['fact'] = y_te.values
preds_df['pred'] = preds_test

fig = px.scatter(
    preds_df, x='fact', y='pred',
    opacity=0.6,
    labels={'fact': 'Факт', 'pred': 'Прогноз'},
    title=f'Калибровка финальной модели — R²={r2:.3f}',
    trendline='ols',
    color_discrete_sequence=['#4363d8'],
)
max_val = max(preds_df['fact'].max(), preds_df['pred'].max()) + 5
fig.add_trace(go.Scatter(
    x=[0, max_val], y=[0, max_val],
    mode='lines', name='Идеал',
    line=dict(color='#e6194b', dash='dash', width=1.5),
))
fig.update_layout(height=450, plot_bgcolor='white')
fig.show()